#  분자에 대한 비지도 임베딩(Unsupervised Embedding) 학습하기

이번 튜토리얼에서는 `SeqToSeq` 모델을 사용해 분자를 분류하기 위한 지문(fingerprint)을 생성해 보겠습니다. 이는 다음 논문을 기반으로 하되, 구현 세부 사항은 일부 다릅니다: Xu et al., "Seq2seq Fingerprint: An Unsupervised Deep Molecular Embedding for Drug Discovery" (https://doi.org/10.1145/3107411.3107424).

## Kaggle에서 실행하기

이 노트북은 Kaggle에서 실행할 수 있습니다. 아래 배지를 클릭하면 Kaggle에서 바로 열립니다.

[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https://github.com/isg-yhlee93/lecture/blob/main/Molecular%20Machine%20Learning/3_Learning_Unsupervised_Embeddings_for_Molecules.ipynb)


In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # DeepChem은 Keras 2 기반이므로 레거시 Keras 사용 (Keras 3 비호환 회피)

!pip install -qq --pre deepchem tf_keras   # tf_keras = Keras 2 백엔드 제공
import deepchem

# 불필요한 경고·로그 메시지 끄기 (화면을 깔끔하게 보기 위함)
import warnings
warnings.filterwarnings('ignore')   # 파이썬 경고 숨기기

from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')      # RDKit 로그 끄기

deepchem.__version__


# SeqToSeq로 임베딩(Embedding) 학습하기

많은 종류의 모델은 입력이 고정된 형태(shape)를 가져야 합니다. 그런데 분자는 포함하는 원자(atom)와 결합(bond)의 수가 매우 다양하기 때문에, 이런 모델을 분자에 적용하기가 어렵습니다. 따라서 각 분자에 대해 고정 길이의 "지문(fingerprint)"을 생성하는 방법이 필요합니다. 이를 위한 다양한 방법이 고안되어 왔으며, 앞선 튜토리얼에서 사용한 확장 연결성 지문(ECFP)도 그중 하나입니다. 하지만 이번 예제에서는 지문을 사람이 직접 설계하는 대신, `SeqToSeq` 모델이 지문을 만드는 방법을 **스스로 학습**하도록 하겠습니다.

`SeqToSeq` 모델은 시퀀스-투-시퀀스(sequence to sequence) 번역을 수행합니다. 예를 들어 한 언어의 텍스트를 다른 언어로 번역하는 데 자주 사용됩니다. 이 모델은 "인코더(encoder)"와 "디코더(decoder)"라는 두 부분으로 이루어집니다. 인코더는 순환 층(recurrent layer)을 쌓은 구조입니다. 입력 시퀀스가 토큰 단위로 하나씩 인코더에 들어가면, "임베딩 벡터(embedding vector)"라고 부르는 고정 길이 벡터를 생성합니다. 디코더는 또 다른 순환 층 더미로, 그 반대 작업을 수행합니다. 즉 임베딩 벡터를 입력으로 받아 출력 시퀀스를 생성합니다. 적절히 선택한 입력/출력 쌍으로 학습시키면, 여러 종류의 변환을 수행하는 모델을 만들 수 있습니다.

여기서는 분자를 나타내는 SMILES 문자열을 입력 시퀀스로 사용합니다. 모델을 오토인코더(autoencoder)로 학습시켜, 출력 시퀀스가 입력 시퀀스와 똑같아지도록 만들 것입니다. 이것이 제대로 동작하려면 인코더는 원래 시퀀스의 모든 정보를 담은 임베딩 벡터를 만들어야 합니다. 그리고 그것이 바로 우리가 지문에서 원하는 바입니다. 따라서 이렇게 얻은 임베딩 벡터는 다른 모델에서 분자를 표현하는 방법으로도 유용할 것입니다!

데이터를 불러오는 것부터 시작하겠습니다. 우리는 MUV 데이터셋을 사용합니다. 이 데이터셋은 학습 세트에 74,501개, 검증 세트에 9,313개의 분자를 포함하고 있어, 다룰 SMILES 문자열이 충분히 많습니다.


In [1]:
import deepchem as dc
# MUV 데이터셋을 stratified 방식으로 분할하여 불러옵니다.
tasks, datasets, transformers = dc.molnet.load_muv(split='stratified')
train_dataset, valid_dataset, test_dataset = datasets
# 각 데이터셋의 SMILES 문자열을 가져옵니다.
train_smiles = train_dataset.ids
valid_smiles = valid_dataset.ids


`SeqToSeq` 모델을 위한 "알파벳(alphabet)", 즉 시퀀스에 나타날 수 있는 모든 토큰의 목록을 정의해야 합니다. (입력 시퀀스와 출력 시퀀스가 서로 다른 알파벳을 가질 수도 있지만, 여기서는 오토인코더로 학습하므로 둘이 동일합니다.) 학습 시퀀스에 등장하는 모든 문자(character)의 목록을 만들어 봅시다.


In [2]:
tokens = set()
# 모든 학습용 SMILES에 등장하는 문자를 모읍니다.
for s in train_smiles:
  tokens = tokens.union(set(c for c in s))
tokens = sorted(list(tokens))


모델을 생성하고 사용할 최적화(optimization) 방법을 정의합니다. 이 경우 학습률(learning rate)을 점진적으로 낮추면 학습이 훨씬 더 잘 됩니다. 매 에폭(epoch)마다 학습률에 0.9를 곱하기 위해 `ExponentialDecay`를 사용합니다.


In [3]:
from deepchem.models.optimizers import Adam, ExponentialDecay
max_length = max(len(s) for s in train_smiles)   # 가장 긴 SMILES 길이
batch_size = 100
batches_per_epoch = len(train_smiles)/batch_size
# 인코더·디코더 각각 2개 층, 256차원 임베딩을 갖는 SeqToSeq 모델을 만듭니다.
model = dc.models.SeqToSeq(tokens,
                           tokens,
                           max_length,
                           encoder_layers=2,
                           decoder_layers=2,
                           embedding_dimension=256,
                           model_dir='fingerprint',
                           batch_size=batch_size,
                           learning_rate=ExponentialDecay(0.001, 0.9, batches_per_epoch))


이제 학습시켜 봅시다! `fit_sequences()`의 입력은 입력/출력 쌍을 만들어내는 제너레이터(generator)입니다. 좋은 GPU에서는 몇 시간 이내에 끝납니다.


In [4]:
def generate_sequences(epochs):
  # 오토인코더이므로 입력과 출력이 동일한 (s, s) 쌍을 만들어냅니다.
  for i in range(epochs):
    for s in train_smiles:
      yield (s, s)

# 40 에폭 동안 학습시킵니다.
model.fit_sequences(generate_sequences(40))


오토인코더로서 얼마나 잘 동작하는지 확인해 봅시다. 검증 세트의 처음 500개 분자를 모델에 통과시켜, 그중 몇 개가 정확히 재현되는지 보겠습니다.


In [5]:
predicted = model.predict_from_sequences(valid_smiles[:500])
count = 0
for s,p in zip(valid_smiles[:500], predicted):
  if ''.join(p) == s:   # 출력이 입력과 정확히 같은지 확인
    count += 1
print('500개의 검증 SMILES 중', count, '개를 정확히 재현했습니다')


reproduced 161 of 500 validation SMILES strings


이제 인코더를 분자 지문 생성기로 사용해 보겠습니다. 학습 및 검증 데이터셋의 모든 분자에 대해 임베딩 벡터를 계산하고, 그 벡터를 특징 벡터(feature vector)로 갖는 새로운 데이터셋을 만듭니다. 데이터 양이 충분히 적어서 모든 것을 메모리에 저장할 수 있습니다.


In [6]:
import numpy as np
# 인코더로 각 분자의 임베딩 벡터(256차원 지문)를 계산합니다.
train_embeddings = model.predict_embeddings(train_smiles)
train_embeddings_dataset = dc.data.NumpyDataset(train_embeddings,
                                                train_dataset.y,
                                                train_dataset.w.astype(np.float32),
                                                train_dataset.ids)

valid_embeddings = model.predict_embeddings(valid_smiles)
valid_embeddings_dataset = dc.data.NumpyDataset(valid_embeddings,
                                                valid_dataset.y,
                                                valid_dataset.w.astype(np.float32),
                                                valid_dataset.ids)


분류(classification)에는 은닉층(hidden layer)이 하나인 단순한 완전 연결 신경망을 사용하겠습니다.


In [7]:
# 임베딩(256차원)을 입력으로 받아 각 작업(task)을 분류하는 모델을 만듭니다.
classifier = dc.models.MultitaskClassifier(n_tasks=len(tasks),
                                                      n_features=256,
                                                      layer_sizes=[512])
classifier.fit(train_embeddings_dataset, nb_epoch=10)


0.0014195525646209716

얼마나 잘 동작하는지 확인해 봅시다. 학습 및 검증 데이터셋에 대한 ROC AUC를 계산합니다.


In [8]:
metric = dc.metrics.Metric(dc.metrics.roc_auc_score, np.mean, mode="classification")
train_score = classifier.evaluate(train_embeddings_dataset, [metric], transformers)
valid_score = classifier.evaluate(valid_embeddings_dataset, [metric], transformers)
print('학습 세트 ROC AUC:', train_score)
print('검증 세트 ROC AUC:', valid_score)


Training set ROC AUC: {'mean-roc_auc_score': 0.9598792603154332}
Validation set ROC AUC: {'mean-roc_auc_score': 0.7251350862464794}
